# Data Source

Sample data is drawn from the **GeoLife GPS Trajectories** dataset (v1.3), collected by Microsoft Research Asia from 182 users between April 2007 and August 2012. The dataset contains 17,621 GPS trajectories (~1.2M km total distance, 48,000+ hours).

**Download:** https://www.microsoft.com/en-us/download/details.aspx?id=52367

**Required citations:**

[1] Yu Zheng, Lizhu Zhang, Xing Xie, Wei-Ying Ma. Mining interesting locations and travel sequences from GPS trajectories. In *Proceedings of International Conference on World Wide Web (WWW 2009)*, Madrid, Spain. ACM Press: 791–800.

[2] Yu Zheng, Quannan Li, Yukun Chen, Xing Xie, Wei-Ying Ma. Understanding Mobility Based on GPS Data. In *Proceedings of ACM Conference on Ubiquitous Computing (UbiComp 2008)*, Seoul, Korea. ACM Press: 312–321.

[3] Yu Zheng, Xing Xie, Wei-Ying Ma. GeoLife: A Collaborative Social Networking Service among User, Location and Trajectory. Invited paper, *IEEE Data Engineering Bulletin*, 33(2), 2010, pp. 32–40.

In [6]:
# test imports are working
import pandas as pd
import numpy as np
df = pd.DataFrame(np.random.rand(5, 3), columns=['A', 'B', 'C'])
df

,A,B,C
0,0.116031,0.467511,0.361241
1,0.710305,0.660958,0.834216
2,0.882068,0.668611,0.394946
3,0.979207,0.557261,0.097080
4,0.001009,0.720944,0.704752


## Bronze Layer — Raw Ingestion

The bronze layer (`backend/pipeline/ingest/geolife.py`) parses the raw GeoLife `.plt` trajectory files into one Parquet file per user under `database/bronze/`. Each `.plt` file has a 6-line header followed by rows of `latitude, longitude, _, altitude (feet), days_since_1899, date, time`.

Transformations are intentionally minimal at this stage — only redundant columns (`unused`, `days_since_1899`) are dropped, and `transport_mode` is joined from each user's `labels.txt` where available (mode labels are sparse — only ~69 of 182 users have them).

Bronze is kept **local** (not in Supabase) to stay under the free-tier storage limit and because it's just raw staging data.

**Result:** 24,876,978 raw points across all 182 users. **Runtime:** ~168 seconds (dominated by per-timestamp parsing).

The cells below first explore a single `.plt` file, then load the full bronze dataset back from Parquet.

In [12]:
from pathlib import Path
DATA_DIR = Path("../database/geolife_sample_data_raw")
# test on one file
sample_file = next(DATA_DIR.glob("000/Trajectory/*.plt"))
print(sample_file)
import pandas as pd
df = pd.read_csv(
    sample_file,
    skiprows=6,
    header=None,
    names=["latitude", "longitude", "unused", "altitude_feet", "days_since_1899", "date", "time"]
)
print(df.head())

../database/geolife_sample_data_raw/000/Trajectory/20090412073303.plt
    latitude   longitude  unused  altitude_feet  days_since_1899        date  \
0  40.000017  116.327479       0            105     39915.314618  2009-04-12   
1  40.000168  116.327474       0             80     39915.314688  2009-04-12   
2  40.000055  116.327454       0             99     39915.314745  2009-04-12   
3  40.000021  116.327407       0            109     39915.314803  2009-04-12   
4  40.000035  116.327281       0            111     39915.314861  2009-04-12   

       time  
0  07:33:03  
1  07:33:09  
2  07:33:14  
3  07:33:19  
4  07:33:24  


In [13]:
BRONZE_DIR = Path("../database/bronze")
bronze_df = pd.read_parquet(BRONZE_DIR, 
                            columns = ['geolife_user_id', 'latitude', 'longitude', 
                                        'altitude_feet', 'date_str', 
                                        'time_str', 'transport_mode'],
                            engine='pyarrow')
                            

In [14]:
bronze_df.head()

,geolife_user_id,latitude,longitude,altitude_feet,date_str,time_str,transport_mode
0,000,39.984702,116.318417,492.0,2008-10-23,02:53:04,<NA>
1,000,39.984683,116.318450,492.0,2008-10-23,02:53:10,<NA>
2,000,39.984686,116.318417,492.0,2008-10-23,02:53:15,<NA>
3,000,39.984688,116.318385,492.0,2008-10-23,02:53:20,<NA>
4,000,39.984655,116.318263,492.0,2008-10-23,02:53:25,<NA>


## Silver Layer — Clean & Structured

The silver layer (`backend/pipeline/transform/silver_geolife.py`) reads all bronze Parquet files and produces two cleaned, model-friendly datasets written to `database/silver/`:

- **`traces.parquet`** — one row per GPS point, cleaned and enriched:
  - `recorded_at` parsed from `date_str` + `time_str`
  - `altitude_m` converted from feet
  - invalid coordinates (lat ∉ [-90, 90], lon ∉ [-180, 180]) and duplicate points dropped
  - `speed_mps`, `heading_deg`, and `step_distance_m` derived from consecutive points **within each trip** (first point of a trip is `NaN`)
- **`trips.parquet`** — one row per trip (`.plt` file), aggregated: `started_at`, `ended_at`, `point_count`, `distance_m`, `duration_s`, and `transport_mode`.

**Result:** 24,178,077 traces across 18,670 trips (~700K invalid/duplicate points dropped from bronze). **Runtime:** ~48 seconds.

> **Known data quality note:** raw GPS jitter produces some physically impossible speeds (max ≈ 861,630 m/s) where consecutive points jump position within a 1-second gap. The median speed (~3.8 m/s) is realistic — outliers are filtered downstream in the gold layer / EDA.

In [15]:
SILVER_DIR = Path("../database/silver")

silver_traces = pd.read_parquet(SILVER_DIR / "traces.parquet")
print(f"{len(silver_traces):,} traces")
silver_traces.head()

24,178,077 traces


,geolife_user_id,source_file,latitude,longitude,transport_mode,recorded_at,altitude_m,point_index,step_distance_m,speed_mps,heading_deg
0,000,/Users/maddywang/git/that-way/database/geolife...,39.984702,116.318417,<NA>,2008-10-23 02:53:04,149.9616,0,NaN,NaN,NaN
1,000,/Users/maddywang/git/that-way/database/geolife...,39.984683,116.318450,<NA>,2008-10-23 02:53:10,149.9616,1,3.516886,0.586148,126.922277
2,000,/Users/maddywang/git/that-way/database/geolife...,39.984686,116.318417,<NA>,2008-10-23 02:53:15,149.9616,2,2.831299,0.566260,276.766339
3,000,/Users/maddywang/git/that-way/database/geolife...,39.984688,116.318385,<NA>,2008-10-23 02:53:20,149.9616,3,2.735434,0.547087,274.663284
4,000,/Users/maddywang/git/that-way/database/geolife...,39.984655,116.318263,<NA>,2008-10-23 02:53:25,149.9616,4,11.023008,2.204602,250.555849


In [16]:
silver_trips = pd.read_parquet(SILVER_DIR / "trips.parquet")
print(f"{len(silver_trips):,} trips")
silver_trips.head()

18,670 trips


,source_file,geolife_user_id,transport_mode,started_at,ended_at,point_count,distance_m,duration_s
0,/Users/maddywang/git/that-way/database/geolife...,000,<NA>,2008-10-23 02:53:04,2008-10-23 11:11:12,908,14938.621493,29888.0
1,/Users/maddywang/git/that-way/database/geolife...,000,<NA>,2008-10-24 02:09:59,2008-10-24 02:47:06,244,1303.660058,2227.0
2,/Users/maddywang/git/that-way/database/geolife...,000,<NA>,2008-10-26 13:44:07,2008-10-26 15:04:07,745,18648.325351,4800.0
3,/Users/maddywang/git/that-way/database/geolife...,000,<NA>,2008-10-27 11:54:49,2008-10-27 12:05:54,50,1895.660587,665.0
4,/Users/maddywang/git/that-way/database/geolife...,000,<NA>,2008-10-28 00:38:26,2008-10-28 05:03:42,1477,8598.824070,15916.0


In [17]:
silver_trips["transport_mode"].value_counts(dropna=False)

transport_mode
<NA>        14194
walk         1577
bike          990
bus           740
car           646
subway        248
taxi          201
train          63
airplane        8
boat            2
run             1
Name: count, dtype: Int64

## Gold Layer — Model-Ready Trajectories

The gold layer (`backend/pipeline/transform/gold_training.py`) takes the silver traces and produces the final model-ready dataset at `database/gold/training_trajectories.parquet`. All transport modes are kept — every trip type informs how a user expresses navigation *preferences*.

Enrichment steps:
- **Speed outlier cleaning** — raw GPS jitter produces physically impossible speeds (max ≈ 861,630 m/s). Speeds above 150 m/s are nulled out while the underlying location is preserved, so trajectory geometry stays intact.
- **Trip boundary flags** — `is_trip_start` / `is_trip_end` mark the first and last point of each trip.
- **Trip context join** — every point carries its parent trip's total `trip_distance_m` and `trip_duration_s`.

**Result:** 24,178,077 enriched points across 18,670 trips. After cleaning, speed is realistic (median ~3.8 m/s, max capped at 150). **Runtime:** ~25 seconds.

In [ ]:
GOLD_DIR = Path("../database/gold")

gold = pd.read_parquet(GOLD_DIR / "training_trajectories.parquet")
print(f"{len(gold):,} enriched points across {gold['source_file'].nunique():,} trips")
gold.head()

In [ ]:
# speed distribution after outlier cleaning
gold["speed_mps"].describe()